<a href="https://colab.research.google.com/github/mfletcher011/PYTHON-SCRIPTS-FOR-INVOICES/blob/main/RolexInvoices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Install and import libraries

In [ ]:
!pip install pdfplumber pandas openpyxl
!pip install XlsxWriter

import re
import pandas as pd
import pdfplumber
from google.colab import files

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.5 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.3 MB/s eta 0:00:00


2. Upload PDF file

In [ ]:
uploaded = files.upload()
pdf_paths = [f for f in uploaded if f.lower().endswith('.pdf')]
if not pdf_paths:
    raise ValueError("Upload a PDF invoice.")
pdf_path = pdf_paths[0]

Saving Invoice 3011 (3) (1).pdf to Invoice 3011 (3) (1).pdf


3. Do the magic

In [ ]:

# 2) Gather all lines
all_lines = []
with pdfplumber.open(pdf_path) as pdf:
    for p in pdf.pages:
        all_lines.extend((p.extract_text() or "").split('\n'))

# 3) Build blocks of each item
blocks, cur = [], []
for L in all_lines:
    if L.startswith("Rolex") or L.startswith("Link"):
        if cur: blocks.append(cur)
        cur = [L]
    else:
        if cur: cur.append(L)
if cur: blocks.append(cur)

rows = []
for blk in blocks:
    text = " ".join(blk).replace("  ", " ")

    # try full‐watch line (handles extra “$ ”)
    m = re.search(
      r"Rolex[: ]+(?:Link\s+)?(\S+)\s+DI Price[: ]*\$\s*([\d,]+(?:\.\d{2})?)"
      r".*?Retail[: ]*\$*\s*([\d,]+(?:\.\d{2})?).*?(\d+)\s+\$([\d,]+(?:\.\d{2})?)",
      text, re.IGNORECASE
    )
    # fallback for pure links
    mm = None
    if not m:
        mm = re.search(
          r"Link\s+(\S+).*?Retail[: ]*\$*\s*([\d,]+(?:\.\d{2})?).*?(\d+)\s+\$([\d,]+(?:\.\d{2})?)",
          text, re.IGNORECASE
        )
    if not (m or mm):
        continue

    if m:
        style, dp, rt, q, r = m.groups()
        desc = ""
        if d := re.search(r"Description\s*:\s*(.*?)\s+Serial", text, re.IGNORECASE):
            desc = d.group(1).strip()
        serial = "N/A"
        if s := re.search(r"Serial[#:]?\s*([A-Za-z0-9]+)", text, re.IGNORECASE):
            serial = s.group(1).strip()
    else:
        # Link‐only
        style    = mm.group(1)
        dp       = "0"
        rt, q, r = mm.group(2), mm.group(3), mm.group(4)
        desc     = "Rolex link"
        serial   = "N/A"

    qty   = int(q)
    total = float(r.replace(",", "")) * qty

    rows.append({
        "Style":        style,
        "DI Price":    dp.replace(",", ""),
        "Retail":      rt.replace(",", ""),
        "Quantity":     qty,
        "Rate":        r.replace(",", ""),
        "Total":       total,
        "Serial":      serial,
        "Description": desc
    })

df = pd.DataFrame(rows, columns=[
    "Style","DI Price","Retail","Quantity","Rate","Total","Serial","Description"
])

# convert to true numbers
for c in ["DI Price","Retail","Rate","Total"]:
    df[c] = pd.to_numeric(df[c].replace(",", "", regex=True), errors="coerce").fillna(0)

# export with currency formatting
out = "rolex_invoice_final.xlsx"
with pd.ExcelWriter(out, engine="xlsxwriter") as w:
    df.to_excel(w, index=False, sheet_name="Invoice")
    wb  = w.book
    ws  = w.sheets["Invoice"]
    fmt = wb.add_format({"num_format":"$#,##0.00"})
    for col in [1,2,4,5]:
        ws.set_column(col, col, 15, fmt)

files.download(out)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>